# 📊 03 - 探索性数据分析 (EDA)> "数据自己不会说话，你得替它说"本节用可视化回答以下问题：1. NBA 薪资分布是怎样的？（贫富差距有多大？）2. 不同位置球员的薪资和表现差异？3. 年龄如何影响球员表现和薪资？4. 得分和薪资的关系是什么？

## 1. 加载数据与设置

In [ ]:
import syssys.path.append('..')import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom scripts.utils import save_chart# 加载清洗后数据df = pd.read_csv('../data/processed/nba_clean_2023_24.csv')print(f"数据维度: {df.shape}")print(f"球员数: {df['PLAYER_NAME'].nunique()}")print(f"球队数: {df['TEAM_ABBREVIATION'].nunique()}")# 绘图设置plt.rcParams['figure.dpi'] = 120plt.rcParams['figure.figsize'] = (12, 6)

## 2. 薪资分布分析

In [ ]:
# 薪资分布直方图fig, axes = plt.subplots(1, 2, figsize=(14, 5))# 直方图ax1 = axes[0]salaries_m = df['SALARY'] / 10000  # 万美元ax1.hist(salaries_m, bins=40, edgecolor='white', alpha=0.8, color='steelblue')ax1.axvline(salaries_m.median(), color='red', linestyle='--',             label=f'中位数: ${salaries_m.median():.0f}万')ax1.axvline(salaries_m.mean(), color='orange', linestyle='--',             label=f'平均数: ${salaries_m.mean():.0f}万')ax1.set_xlabel('薪资 (万美元)')ax1.set_ylabel('球员数')ax1.set_title('NBA 球员薪资分布 (2023-24)')ax1.legend()# 按薪资等级ax2 = axes[1]level_counts = df['SALARY_LEVEL'].value_counts()colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#9b59b6']ax2.pie(level_counts.values, labels=level_counts.index, autopct='%1.1f%%',        colors=colors, startangle=90)ax2.set_title('薪资等级构成')plt.tight_layout()save_chart(fig, 'salary_distribution.png')plt.show()# 统计摘要print("📊 薪资分布统计 (万美元):")print(salaries_m.describe().round(0))print(f"\n基尼系数类比 —— Top 10% 球员拿走了总薪资的 {salaries_m.nlargest(int(len(salaries_m)*0.1)).sum() / salaries_m.sum() * 100:.1f}%")

## 3. 位置与薪资分析

In [ ]:
# 按位置分析（如果有位置数据）fig, ax = plt.subplots(figsize=(10, 6))# 按球队统计平均薪资team_salary = df.groupby('TEAM_ABBREVIATION').agg(    AVG_SALARY=('SALARY', 'mean'),    TOTAL_SALARY=('SALARY', 'sum'),    PLAYER_COUNT=('PLAYER_NAME', 'count')).sort_values('AVG_SALARY', ascending=False)# Top 10 和 Bottom 10top_bottom = pd.concat([team_salary.head(10), team_salary.tail(10)])colors = ['#e74c3c']*10 + ['#2ecc71']*10ax.barh(range(len(top_bottom)), top_bottom['AVG_SALARY']/10000, color=colors)ax.set_yticks(range(len(top_bottom)))ax.set_yticklabels(top_bottom.index)ax.set_xlabel('平均薪资 (万美元)')ax.set_title('球队平均薪资 Top 10 vs Bottom 10')ax.axvline(df['SALARY'].mean()/10000, color='gray', linestyle='--',            label=f'联盟均值: ${df["SALARY"].mean()/10000:.0f}万')plt.tight_layout()save_chart(fig, 'team_salary_comparison.png')plt.show()

## 4. 得分 vs 薪资：钱花对了吗？

In [ ]:
# 散点图：得分 vs 薪资fig, ax = plt.subplots(figsize=(10, 8))scatter = ax.scatter(    df['PTS'], df['SALARY']/10000,    c=df['EFFICIENCY'], cmap='RdYlGn',    s=df['MIN']*2, alpha=0.6, edgecolors='white', linewidth=0.5)# 标注极端值top_5_efficient = df.nlargest(5, 'EFFICIENCY')worst_5 = df.nsmallest(5, 'EFFICIENCY')for _, player in top_5_efficient.iterrows():    ax.annotate(player['PLAYER_NAME'],                 (player['PTS'], player['SALARY']/10000),                fontsize=8, color='darkgreen', fontweight='bold')for _, player in worst_5.iterrows():    ax.annotate(player['PLAYER_NAME'],                (player['PTS'], player['SALARY']/10000),                fontsize=8, color='darkred', fontweight='bold')ax.set_xlabel('场均得分 (PTS)')ax.set_ylabel('薪资 (万美元)')ax.set_title('💰 场均得分 vs 薪资：钱花对了吗？\n(颜色=效率分，大小=出场时间)')cbar = plt.colorbar(scatter)cbar.set_label('效率得分')plt.tight_layout()save_chart(fig, 'pts_vs_salary.png')plt.show()

## 5. 年龄与表现曲线

In [ ]:
# 分析年龄趋势（如果有年龄数据）if 'AGE' in df.columns:    fig, axes = plt.subplots(1, 2, figsize=(14, 5))        # 按年龄分组    df['AGE_GROUP'] = pd.cut(df['AGE'], bins=[18, 22, 25, 28, 31, 35, 45],                              labels=['18-22', '23-25', '26-28', '29-31', '32-35', '36+'])        # 各年龄段场均得分    age_stats = df.groupby('AGE_GROUP', observed=False).agg(        AVG_PTS=('PTS', 'mean'),        AVG_SALARY=('SALARY', 'mean'),        COUNT=('PLAYER_NAME', 'count')    )        ax1 = axes[0]    ax1.bar(age_stats.index.astype(str), age_stats['AVG_PTS'], color='steelblue')    ax1.set_xlabel('年龄段')    ax1.set_ylabel('场均得分')    ax1.set_title('各年龄段场均得分')        ax2 = axes[1]    ax2.bar(age_stats.index.astype(str), age_stats['AVG_SALARY']/10000, color='coral')    ax2.set_xlabel('年龄段')    ax2.set_ylabel('平均薪资 (万美元)')    ax2.set_title('各年龄段平均薪资')        plt.tight_layout()    save_chart(fig, 'age_performance.png')    plt.show()else:    print("⚠️ 数据中无年龄字段 (AGE)，跳过年龄分析")

## 📋 小结探索分析揭示的关键发现：- NBA 薪资呈**严重右偏分布**，少数巨星拉高了均值- 薪资和场上表现**并非严格线性关系**，存在大量效率异常值- 球队薪资结构差异显著，小球市更难吸引高价自由球员下一节：[04_modeling.ipynb](./04_modeling.ipynb) 将用机器学习构建薪资预测模型，精准识别溢价合同。